# 03 — Data Cleaning

## Purpose
Clean and standardize the three raw datasets (Statcast, injury database, player
metadata) and validate that they join cleanly on `player_id` (MLBAM) before
feature engineering begins.

This notebook:
1. Loads raw Statcast, injury, and metadata parquets from notebooks 01/02
2. Cleans Statcast: drops always-null columns, deduplicates on pitch key, standardizes pitch type codes, removes implausible sensor values
3. Cleans the injury database: audits `other` stints, re-applies updated injury type patterns, flags retroactive short stints
4. Supplements player metadata with birth dates from the MLB Stats API (Chadwick register omits them)
5. Validates cross-source join coverage (Statcast ↔ metadata ↔ injury DB)
6. Saves cleaned outputs to `data/processed/`

## Outputs
- `data/processed/statcast_clean.parquet`
- `data/processed/injuries_clean.parquet`
- `data/processed/player_metadata_clean.parquet`

In [ ]:
import importlib
import json
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore", category=FutureWarning)

# Ensure project root is on sys.path so src/ imports work.
PROJECT_ROOT = str(Path(".").resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Reload so changes to injury_loader.py take effect without a kernel restart.
import src.data.injury_loader as _il_mod
importlib.reload(_il_mod)
from src.data.injury_loader import parse_injury_type
print("injury_loader loaded from:", _il_mod.__file__)

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

---
## 1 — Load Raw Data

In [ ]:
# ── Statcast: load all parquet files under data/raw/statcast/ ────────────────
sc_files = sorted(Path("data/raw/statcast").glob("*.parquet"))
if not sc_files:
    raise FileNotFoundError("No Statcast parquet files found. Run notebook 01 first.")
sc_raw = pd.concat([pd.read_parquet(f) for f in sc_files], ignore_index=True)
sc_raw["game_date"] = pd.to_datetime(sc_raw["game_date"])
print(f"Statcast raw:  {sc_raw.shape[0]:>7,} rows  |  {sc_raw.shape[1]} columns  |  "
      f"{sc_raw['game_date'].dt.year.unique().tolist()} seasons")

# ── Injury database ─────────────────────────────────────────────────────────
inj_path = Path("data/raw/injuries/injury_database.parquet")
if not inj_path.exists():
    raise FileNotFoundError("No injury database found. Run notebook 02 first.")
inj_raw = pd.read_parquet(inj_path)
print(f"Injury DB raw: {len(inj_raw):>7,} stints  |  {inj_raw['player_id'].nunique()} pitchers")

# ── Player metadata ──────────────────────────────────────────────────────────
meta_path = Path("data/raw/player_metadata/pitchers.parquet")
if not meta_path.exists():
    raise FileNotFoundError("No player metadata found. Run notebook 01 first.")
meta_raw = pd.read_parquet(meta_path)
print(f"Metadata raw:  {len(meta_raw):>7,} pitchers  |  columns: {meta_raw.columns.tolist()}")

---
## 2 — Statcast Cleaning

Steps applied in order:
1. Drop columns that are 100% null (no information content)
2. Deduplicate on the natural pitch key `(game_pk, at_bat_number, pitch_number)`
3. Standardize pitch type codes across eras (e.g. FT→SI, FA→FF from pre-2019 data)
4. Flag and remove rows with physically implausible values
5. Summarize null rates for modeling-relevant columns

In [ ]:
# ── 2a: Drop always-null columns ─────────────────────────────────────────────
# spin_dir and sv_id are always null in pybaseball pulls and carry no signal.
always_null = [c for c in sc_raw.columns if sc_raw[c].isna().all()]
print(f"Always-null columns ({len(always_null)}): {always_null}")

# Also drop columns that are >99% null — they can't be useful features.
null_rate = sc_raw.isna().mean()
mostly_null = null_rate[null_rate > 0.99].index.tolist()
# Remove already-flagged duplicates.
mostly_null = [c for c in mostly_null if c not in always_null]
print(f"Mostly-null columns (>99%) ({len(mostly_null)}): {mostly_null}")

drop_cols = always_null + mostly_null
sc = sc_raw.drop(columns=drop_cols)
print(f"\nColumns after drop: {sc_raw.shape[1]} → {sc.shape[1]}")

In [ ]:
# ── 2b: Deduplication ────────────────────────────────────────────────────────
# A pitch is uniquely identified by (game_pk, at_bat_number, pitch_number).
# Duplicates arise when multiple parquet files overlap in date range.
PITCH_KEY = ["game_pk", "at_bat_number", "pitch_number"]
key_cols_present = [c for c in PITCH_KEY if c in sc.columns]

n_before = len(sc)
if len(key_cols_present) == len(PITCH_KEY):
    sc = sc.drop_duplicates(subset=PITCH_KEY)
    n_dropped = n_before - len(sc)
    print(f"Duplicate pitches removed: {n_dropped:,}  ({n_before:,} → {len(sc):,})")
else:
    missing = set(PITCH_KEY) - set(key_cols_present)
    print(f"WARNING: dedup skipped — missing key columns: {missing}")
    n_dropped = 0

sc = sc.reset_index(drop=True)
print(f"Rows after deduplication: {len(sc):,}")

In [ ]:
# ── 2c: Pitch type standardization ───────────────────────────────────────────
# Remap legacy codes (FT→SI, FA→FF, FO→FS, CS→CU) and drop non-pitch rows.

PITCH_TYPE_REMAP = {"FT": "SI", "FA": "FF", "FO": "FS", "CS": "CU"}
PITCH_TYPES_DROP = {"PO", "IN", "AB"}

print("Pitch type distribution BEFORE standardization:")
print(sc["pitch_type"].value_counts().to_string())

n_before = len(sc)
sc["pitch_type"] = sc["pitch_type"].replace(PITCH_TYPE_REMAP)
sc = sc[~sc["pitch_type"].isin(PITCH_TYPES_DROP)]

print(f"\nRows removed (pitchouts/intentionals): {n_before - len(sc):,}")
print("\nPitch type distribution AFTER standardization:")
print(sc["pitch_type"].value_counts().to_string())

In [ ]:
# ── 2d: Implausible value check ───────────────────────────────────────────────
# Flag rows outside physical sensor bounds (speed 40–105 mph, spin 1000–4000 rpm, etc.).

BOUNDS = {
    "release_speed":     (40,   105),
    "release_spin_rate": (1000, 4000),
    "release_extension": (3.0,  8.5),
    "pfx_x":             (-3.0, 3.0),
    "pfx_z":             (-3.0, 3.0),
}

outlier_mask = pd.Series(False, index=sc.index)
print("Implausible value counts (rows that violate bounds):")
for col, (lo, hi) in BOUNDS.items():
    if col not in sc.columns:
        print(f"  {col}: column not present, skipping")
        continue
    bad = sc[col].notna() & ((sc[col] < lo) | (sc[col] > hi))
    outlier_mask |= bad
    if bad.sum() > 0:
        print(f"  {col} [{lo}–{hi}]:  {bad.sum()} outliers")
        print(sc.loc[bad, ["pitcher", "player_name", "game_date", col]].head(5).to_string())
    else:
        print(f"  {col} [{lo}–{hi}]:  OK")

n_before = len(sc)
sc = sc[~outlier_mask].reset_index(drop=True)
print(f"\nRows removed (implausible values): {n_before - len(sc):,}")
print(f"Statcast rows remaining: {len(sc):,}")

In [ ]:
# ── 2e: Null rate summary for modeling-relevant columns ───────────────────────
# Non-ball-in-play nulls are expected; flag any modeling column above 5%.

MODELING_COLS = [
    "release_speed", "release_spin_rate", "release_extension",
    "pfx_x", "pfx_z", "plate_x", "plate_z",
    "release_pos_x", "release_pos_y", "release_pos_z",
    "effective_speed", "spin_axis",
    "pitch_type", "game_date", "pitcher",
]

present = [c for c in MODELING_COLS if c in sc.columns]
null_rates = sc[present].isna().mean().sort_values(ascending=False)
print("Null rates for modeling-relevant columns:")
for col, rate in null_rates.items():
    flag = "  ← review" if rate > 0.05 else ""
    print(f"  {col:<25}  {rate:.1%}{flag}")

---
## 3 — Injury Database Cleaning

Steps:
1. Audit stints classified as `other` — identify fixable gaps in the regex patterns
2. Re-apply `parse_injury_type` from the updated `injury_loader.py`
3. Review the days-lost distribution (median should be ~15–25 days)
4. Flag retroactive short-stints (<10 days) — these are real but unusual

In [ ]:
# ── 3a: "Other" injury type audit (on the raw classifications) ────────────────
# Show all stints classified as 'other' and whether the description contains
# any recognisable body-part keyword that the old patterns missed.

print(f"Injury type distribution (raw, before re-classification):")
print(inj_raw["injury_type"].value_counts().to_string())
print()

others = inj_raw[inj_raw["injury_type"] == "other"].copy()
print(f"'Other' stints: {len(others)} of {len(inj_raw)} ({len(others)/len(inj_raw):.1%})")
print()

# Separate into: has a description with body-part text vs. truly undescribed.
has_desc = others[others["description"].str.contains(
    r"strain|contusion|inflam|fracture|tendin|sprain|tightness|soreness|pain|"
    r"fatigue|surgery|procedure|tear|rupture",
    case=False, na=False, regex=True
)]
no_desc = others[~others.index.isin(has_desc.index)]

print(f"  Has injury text in description: {len(has_desc)}  ← fixable via pattern update")
print(f"  No useful description:          {len(no_desc)}  ← irreducible (API omitted details)")
print()
print("Descriptions with injury text (showing first 20):")
for _, row in has_desc.head(20).iterrows():
    print(f"  {row.player_name}: {row.description}")

In [ ]:
# ── 3b: Re-apply updated injury classification ────────────────────────────────
# Re-run parse_injury_type with expanded patterns to reduce 'other' stints.

inj = inj_raw.copy()
inj["injury_type"] = inj["description"].apply(parse_injury_type)

print("Injury type distribution BEFORE re-classification:")
before = inj_raw["injury_type"].value_counts()
print(before.to_string())
print()
print("Injury type distribution AFTER re-classification:")
after = inj["injury_type"].value_counts()
print(after.to_string())
print()

# Show the delta — which categories gained stints.
delta = after.subtract(before, fill_value=0).astype(int)
delta = delta[delta != 0].sort_values(ascending=False)
print("Net change per category (+ = gained, − = reduced):")
for cat, d in delta.items():
    sign = "+" if d > 0 else ""
    print(f"  {cat:<15}  {sign}{d}")

In [ ]:
# ── 3c: Days-lost distribution ────────────────────────────────────────────────
# Expected shape: right-skewed with a floor around 10 days (minimum IL stint),
# a mode around 15-25 days, and a long tail for season-ending injuries.

paired = inj[inj["days_lost"].notna()]
season_ending = inj[inj["season_ending"] == True]

print(f"Total stints:          {len(inj):>5}")
print(f"Paired (days_lost):    {len(paired):>5}  ({len(paired)/len(inj):.1%})")
print(f"Season-ending (no act):{len(season_ending):>5}  ({len(season_ending)/len(inj):.1%})")
print()
print("Days-lost summary (paired stints only):")
print(paired["days_lost"].describe().to_frame().T.to_string())
print()

# Days-lost by injury type (median — easier to interpret than mean for skewed data).
print("Median days lost by injury type:")
medians = (
    paired.groupby("injury_type")["days_lost"]
    .agg(["median", "count"])
    .sort_values("median", ascending=False)
)
medians.columns = ["median_days", "n_stints"]
print(medians.to_string())

In [ ]:
# ── 3d: Flag short/retroactive stints (<10 days) ─────────────────────────────
# End-of-season retroactive IL moves legitimately show <10 days; flag but keep.

SHORT_THRESH = 10  # days

inj["retroactive_short"] = (
    inj["days_lost"].notna()
    & (inj["days_lost"] < SHORT_THRESH)
)

short = inj[inj["retroactive_short"]]
print(f"Stints under {SHORT_THRESH} days: {len(short)} of {len(paired)} paired stints")
print()
print(short[["player_name", "transaction_date", "activation_date",
             "days_lost", "injury_type", "description"]].to_string())

---
## 4 — Player Metadata Cleaning

The Chadwick Bureau register (used in notebook 01) returns only cross-reference IDs
(MLBAM, FanGraphs, BBRef). Birth dates are not included and came back as `NaT`.
Here we supplement from the MLB Stats API, which returns birth dates for all players
in one batch request.

In [ ]:
# ── 4a: ID completeness ───────────────────────────────────────────────────────
meta = meta_raw.copy()

print(f"Total pitchers in metadata: {len(meta)}")
print()
print("Null rates per column:")
for col in meta.columns:
    rate = meta[col].isna().mean()
    flag = "  ← all null" if rate == 1.0 else ("  ← review" if rate > 0.1 else "")
    print(f"  {col:<20}  {rate:.1%}{flag}")

# ID completeness: how many pitchers have all three cross-reference IDs?
has_all_ids = meta[["player_id", "key_fangraphs", "key_bbref"]].notna().all(axis=1)
print(f"\nPitchers with all 3 IDs (MLBAM + FanGraphs + BBRef): {has_all_ids.sum()} / {len(meta)}")

In [ ]:
# ── 4b: Supplement birth dates from MLB Stats API ─────────────────────────────
# Batch fetch birth dates at 200 IDs per request to avoid URL length limits.

BATCH_SIZE = 200
BASE_URL = "https://statsapi.mlb.com/api/v1/people"

mlbam_ids = meta["player_id"].dropna().astype(int).tolist()
birth_map: dict[int, str] = {}

print(f"Fetching birth dates for {len(mlbam_ids)} pitchers in "
      f"{(len(mlbam_ids) + BATCH_SIZE - 1) // BATCH_SIZE} batches…")

for i in range(0, len(mlbam_ids), BATCH_SIZE):
    batch = mlbam_ids[i : i + BATCH_SIZE]
    params = {"personIds": ",".join(str(x) for x in batch),
              "fields": "people,id,birthDate"}
    try:
        resp = requests.get(BASE_URL, params=params, timeout=30)
        resp.raise_for_status()
        for person in resp.json().get("people", []):
            pid = person.get("id")
            bdate = person.get("birthDate")
            if pid and bdate:
                birth_map[int(pid)] = bdate
    except Exception as exc:
        print(f"  Batch {i//BATCH_SIZE + 1} failed: {exc}")
    time.sleep(0.5)

print(f"Birth dates retrieved: {len(birth_map)} / {len(mlbam_ids)}")

birth_series = meta["player_id"].map(birth_map)
meta["birth_date"] = pd.to_datetime(birth_series, errors="coerce")

still_null = meta["birth_date"].isna().sum()
print(f"Birth dates still null after supplement: {still_null}")
print(meta[["player_name", "birth_date"]].dropna().head(10).to_string())

---
## 5 — Cross-Source Join Validation

Checks that the three datasets connect cleanly on `player_id` (MLBAM).

**Note on TEST_MODE:** In TEST_MODE we have only 1 week of Statcast (June 1–7 2023)
but a full 2023 season of injuries. A pitcher who was injured in March will not appear
in the June Statcast slice. The low overlap here is *expected* — it will resolve when
the full season Statcast data is pulled.

In [ ]:
# ── 5a: Statcast pitchers ↔ metadata ─────────────────────────────────────────
sc_pitchers = set(sc["pitcher"].dropna().astype(int).unique())
meta_pitchers = set(meta["player_id"].dropna().astype(int).unique())

in_sc_and_meta = sc_pitchers & meta_pitchers
in_sc_not_meta = sc_pitchers - meta_pitchers
in_meta_not_sc = meta_pitchers - sc_pitchers

print(f"Statcast unique pitchers:        {len(sc_pitchers):>5}")
print(f"Metadata unique pitchers:        {len(meta_pitchers):>5}")
print(f"In both (Statcast ∩ metadata):   {len(in_sc_and_meta):>5}  "
      f"({len(in_sc_and_meta)/len(sc_pitchers):.1%} of Statcast pitchers)")
print(f"In Statcast, not in metadata:    {len(in_sc_not_meta):>5}")
print(f"In metadata, not in Statcast:    {len(in_meta_not_sc):>5}  "
      f"(expected — metadata covers full season; Statcast is TEST_MODE slice)")

if in_sc_not_meta:
    # These pitchers appeared in a game but aren't in our metadata — flag them.
    print(f"\nStatcast pitchers missing from metadata (sample):")
    missing_names = sc[sc["pitcher"].isin(in_sc_not_meta)][["pitcher", "player_name"]]\
        .drop_duplicates().head(10)
    print(missing_names.to_string())

In [ ]:
# ── 5b: Statcast pitchers ↔ injury database ───────────────────────────────────
inj_pitchers = set(inj["player_id"].dropna().astype(int).unique())

in_both   = sc_pitchers & inj_pitchers
inj_only  = inj_pitchers - sc_pitchers
sc_only   = sc_pitchers - inj_pitchers

print(f"Statcast unique pitchers:        {len(sc_pitchers):>5}")
print(f"Injury DB unique pitchers:       {len(inj_pitchers):>5}")
print(f"In both:                         {len(in_both):>5}")
print(f"Injured, not in Statcast slice:  {len(inj_only):>5}  "
      f"(normal — most injuries happened outside June 1-7)")
print(f"In Statcast, not injured:        {len(sc_only):>5}  "
      f"(pitchers who stayed healthy this week)")

if in_both:
    print(f"\nPitchers found in both Statcast and injury DB (sample):")
    both_names = inj[inj["player_id"].isin(in_both)][["player_id", "player_name",
                                                        "injury_type", "transaction_date"]]\
        .drop_duplicates("player_id").head(10)
    print(both_names.to_string())

In [ ]:
# ── 5c: Injury DB pitchers ↔ metadata ────────────────────────────────────────
inj_in_meta  = inj_pitchers & meta_pitchers
inj_not_meta = inj_pitchers - meta_pitchers

print(f"Injured pitchers in metadata:    {len(inj_in_meta):>5}  "
      f"({len(inj_in_meta)/len(inj_pitchers):.1%} of injured pitchers)")
print(f"Injured pitchers not in metadata:{len(inj_not_meta):>5}")

if inj_not_meta:
    # Injured pitchers not in metadata are likely two-way players or below pitch threshold.
    print(f"\nInjured pitchers missing from metadata (sample):")
    missing_inj = inj[inj["player_id"].isin(inj_not_meta)][["player_id", "player_name",
                                                              "injury_type"]]\
        .drop_duplicates("player_id").head(10)
    print(missing_inj.to_string())

In [ ]:
# ── 5d: Coverage summary ──────────────────────────────────────────────────────
summary = {
    "Statcast pitches (cleaned)": len(sc),
    "Statcast unique pitchers": len(sc_pitchers),
    "Injury stints (cleaned)": len(inj),
    "Unique injured pitchers": len(inj_pitchers),
    "Pitcher metadata rows": len(meta),
    "Pitchers with birth_date": int(meta["birth_date"].notna().sum()),
    "Statcast ∩ metadata coverage": f"{len(in_sc_and_meta)/len(sc_pitchers):.1%}",
    "Injury DB ∩ metadata coverage": f"{len(inj_in_meta)/len(inj_pitchers):.1%}",
}

print("=" * 52)
print("DATA COVERAGE SUMMARY")
print("=" * 52)
for k, v in summary.items():
    print(f"  {k:<40}  {v}")
print("=" * 52)

---
## 6 — Save Cleaned Outputs

In [ ]:
# ── Save all three cleaned datasets ──────────────────────────────────────────
sc_out   = PROCESSED_DIR / "statcast_clean.parquet"
inj_out  = PROCESSED_DIR / "injuries_clean.parquet"
meta_out = PROCESSED_DIR / "player_metadata_clean.parquet"

sc.to_parquet(sc_out, index=False)
inj.to_parquet(inj_out, index=False)
meta.to_parquet(meta_out, index=False)

print(f"Saved {sc_out}    — {len(sc):,} rows")
print(f"Saved {inj_out}   — {len(inj):,} rows")
print(f"Saved {meta_out}  — {len(meta):,} rows")

In [ ]:
# ── Update provenance.json ────────────────────────────────────────────────────
from datetime import timezone, datetime

prov_path = Path("data/raw/provenance.json")
prov = json.loads(prov_path.read_text()) if prov_path.exists() else {}

prov["cleaning"] = {
    "run_at":               datetime.now(timezone.utc).isoformat(),
    "statcast_rows_raw":    len(sc_raw),
    "statcast_rows_clean":  len(sc),
    "statcast_cols_dropped": drop_cols,
    "injury_stints_raw":    len(inj_raw),
    "injury_stints_clean":  len(inj),
    "injury_other_before":  int(before.get("other", 0)),
    "injury_other_after":   int(after.get("other", 0)),
    "metadata_rows":        len(meta),
    "birth_dates_filled":   int(meta["birth_date"].notna().sum()),
}

prov_path.write_text(json.dumps(prov, indent=2, default=str))
print("Provenance updated:", prov_path)
print(json.dumps(prov["cleaning"], indent=2, default=str))